In [ ]:

# Retrain the two-stage pipeline's Swin-Tiny classifier arm so the pipeline can
# be re-scored on the CORRECTED (leak-free) localiser. Section 4.10 of the paper
# records that the original Swin-Tiny checkpoints were never archived; this is
# the experiment that closes it.
#
# Kaggle traps already hit in this project, guarded here:
#  * P100 is sm_60; recent torch wheels dropped Pascal kernels -> pin cu121
#    BEFORE torch is first imported, and never os.execv to apply it.
#  * ultralytics/timm pull integrations (ray, wandb) whose callbacks raise at an
#    epoch boundary -> remove them first.
import subprocess, sys, os, json, glob, time
info = subprocess.run(["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("GPU:", info or "NONE")
cap = info.split(",")[1].strip() if "," in info else ""
assert "torch" not in sys.modules, "torch already imported"
if cap.startswith("6."):
    print(f"compute capability {cap} is Pascal: pinning torch 2.5.1 + cu121")
    subprocess.run([sys.executable,"-m","pip","-q","install","torch==2.5.1","torchvision==0.20.1",
                    "--index-url","https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run([sys.executable,"-m","pip","-q","uninstall","-y",
                "ray","wandb","comet_ml","mlflow","dvclive","neptune","clearml"], check=False)
subprocess.run([sys.executable,"-m","pip","-q","install","timm==1.0.11"], check=True)
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| arch", torch.cuda.get_arch_list())
assert torch.cuda.is_available()
_x=(torch.randn(64,64,device="cuda")@torch.randn(64,64,device="cuda")).sum().item(); torch.cuda.synchronize()
print("CUDA smoke test PASSED", round(_x,3))


In [ ]:

# The classifier is trained on GROUND-TRUTH masked images (as in the original
# benchmark) and evaluated on masks PREDICTED by the retrained leak-free
# localiser -- that asymmetry is the deployed protocol and is preserved exactly.
MASKED = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("masked") and {"train","valid","test"} <= set(dns):
        MASKED = dp; break
assert MASKED, "ground-truth masked classification dataset not found"
PRED = None
for dp, dns, fns in os.walk("/kaggle/input"):
    if dp.rstrip("/").endswith("masked_test"):
        PRED = dp; break
assert PRED, "predicted-mask test images not found"
print("GT-masked train dir:", MASKED)
print("predicted-mask test:", PRED, len(glob.glob(PRED+"/*")), "images")
for sp in ("train","valid","test"):
    print(" ", sp, len(glob.glob(f"{MASKED}/{sp}/*/*")))


In [ ]:

import numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets
from PIL import Image

IMG=224
tf_train = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2), transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_eval = transforms.Compose([
    transforms.Resize((IMG,IMG)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = datasets.ImageFolder(f"{MASKED}/train", tf_train)
val_ds   = datasets.ImageFolder(f"{MASKED}/valid", tf_eval)
print("classes:", train_ds.classes)

# Ground truth for the predicted-mask test set, taken from the released
# per-image predictions so the labels match the paper exactly.
gt = {r["img"]: r["true"] for r in json.load(open(
      glob.glob("/kaggle/input/**/preds_internal.json", recursive=True)[0]))} \
     if glob.glob("/kaggle/input/**/preds_internal.json", recursive=True) else None
if gt is None:
    # fall back: label from the GT-masked test folder structure
    gt = {}
    for c in sorted(os.listdir(f"{MASKED}/test")):
        for p in glob.glob(f"{MASKED}/test/{c}/*"): gt[os.path.basename(p)] = int(c)
print("test labels available:", len(gt))

class PredMaskTest(Dataset):
    def __init__(self, d, gt, tf):
        self.items=[(p, gt[os.path.basename(p)]) for p in sorted(glob.glob(d+"/*"))
                    if os.path.basename(p) in gt]
        self.tf=tf
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,y=self.items[i]; return self.tf(Image.open(p).convert("RGB")), y, os.path.basename(p)
test_ds = PredMaskTest(PRED, gt, tf_eval)
print("predicted-mask test images matched to a label:", len(test_ds))

cnt=np.bincount([y for _,y in train_ds.samples], minlength=3).astype(float)
w=torch.tensor(cnt.sum()/(3*cnt), dtype=torch.float32).cuda()
print("class weights:", w.tolist())

def run_seed(seed, batch, lr, epochs=30, patience=7):
    torch.manual_seed(seed); np.random.seed(seed)
    tl=DataLoader(train_ds,batch_size=batch,shuffle=True,num_workers=2,drop_last=False)
    vl=DataLoader(val_ds,batch_size=64,num_workers=2)
    el=DataLoader(test_ds,batch_size=64,num_workers=2)
    m=timm.create_model("swin_tiny_patch4_window7_224",pretrained=True,num_classes=3).cuda()
    opt=torch.optim.AdamW(m.parameters(),lr=lr,weight_decay=0.05)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    crit=nn.CrossEntropyLoss(weight=w); scaler=torch.amp.GradScaler("cuda")
    best,bad,bstate=-1,0,None
    for ep in range(epochs):
        m.train()
        for x,y in tl:
            x,y=x.cuda(non_blocking=True),y.cuda(non_blocking=True); opt.zero_grad()
            with torch.amp.autocast("cuda"): loss=crit(m(x),y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sch.step()
        m.eval(); c=t=0
        with torch.no_grad():
            for x,y in vl:
                with torch.amp.autocast("cuda"): p=m(x.cuda()).argmax(1).cpu()
                c+=(p==y).sum().item(); t+=len(y)
        va=100*c/t
        if va>best: best,bad,bstate=va,0,{k:v.detach().clone() for k,v in m.state_dict().items()}
        else:
            bad+=1
            if bad>=patience: break
    m.load_state_dict(bstate); m.eval()
    preds,labels,names=[],[],[]
    with torch.no_grad():
        for x,y,nm in el:
            with torch.amp.autocast("cuda"): p=m(x.cuda()).argmax(1).cpu()
            preds+=p.tolist(); labels+=y.tolist(); names+=list(nm)
    acc=100*sum(int(a==b) for a,b in zip(preds,labels))/len(labels)
    per={}
    for a,b in zip(preds,labels):
        per.setdefault(b,[0,0]); per[b][1]+=1
        if a==b: per[b][0]+=1
    bal=100*float(np.mean([c/t for c,t in per.values()]))
    return {"seed":seed,"batch":batch,"lr":lr,"best_val":best,"int_acc":acc,
            "int_bal":bal,"preds":preds,"labels":labels,"imgs":names}

# The ORIGINAL per-seed recipe is preserved exactly, so the only thing that
# changes versus the published pipeline arm is the localiser that made the masks.
RECIPE=[(s,16,1e-4) for s in range(7)]+[(s,32,2e-4) for s in (7,8,9)]
out=[]
for i,(s,b,lr) in enumerate(RECIPE):
    t0=time.time(); r=run_seed(s,b,lr); r["minutes"]=round((time.time()-t0)/60,1)
    out.append(r)
    json.dump(out, open("/kaggle/working/pipeline_arm_leakfree_localiser.json","w"), indent=1)
    print(f"[{i+1}/10] seed {s} batch {b} lr {lr}: pipeline internal "
          f"{r['int_acc']:.2f}%  balanced {r['int_bal']:.2f}%  ({r['minutes']} min)")
a=np.array([r["int_acc"] for r in out])
print(f"\nPIPELINE ARM on the CORRECTED localiser: mean {a.mean():.2f} +- {a.std(ddof=1):.2f}")
print(f"Published (contaminated localiser) was: 80.98 +- 2.22")
